# Gemini ANY Mode: Enum Complexity Causes Opaque `INVALID_ARGUMENT`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/radishbuild/field-notes/blob/main/gemini-any-mode-enum-bug/repro_minimal.ipynb)

## What this reproduces

When using Gemini's `FunctionCallingConfig(mode="ANY")`, the API returns an opaque `INVALID_ARGUMENT` error when tool schemas contain many enum fields with multi-token values. **AUTO mode with identical schemas works fine.**

This notebook demonstrates that the failure depends on **enum string content**, not schema structure. I generate identical schemas (same number of tools, enum fields, and values per enum) and only vary the enum strings:

- **1-word values** (`"action"`, `"pending"`) → **PASS**
- **2-word values** (`"action_required"`, `"deploy_failed"`) → **FAIL**
- **3-word values** (`"action_required_review"`) → **FAIL**
- **4-word values** (`"action_required_review_pending"`) → **FAIL**

## Hypothesis

I suspect ANY mode hits some opaque internal budget limit — likely related to token count or schema complexity — that is undocumented and produces zero diagnostic information. Multi-token compound strings (like `action_required`) consume more of this budget than single-token values, causing the request to be rejected with a bare `INVALID_ARGUMENT` error once the budget is exceeded. The longer the enum names, the sooner the limit is reached — 3-4 word enum values start failing at much lower total counts than 1-2 word values with the same number of tools and fields.

In [ ]:
!pip install -q google-genai

## API Key Setup

Go to **Colab menu → 🔑 Secrets** (key icon in left sidebar), add a secret named `GEMINI_API_KEY` with your [Gemini API key](https://aistudio.google.com/apikey), and enable notebook access.

In [2]:
import os
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except ImportError:
    api_key = os.environ["GEMINI_API_KEY"]

client = genai.Client(api_key=api_key)
print("Client ready ✓")

Client ready ✓


## Configuration

In [3]:
MODEL = "gemini-3-flash-preview"

## Word List & Pool Generation

I generate pools of enum values with 1, 2, 3, and 4 words joined by underscores. A seeded RNG picks random word combinations to avoid prefix sharing.

In [4]:
import random
import json

WORDS = [
    "action", "required", "pending", "completed", "triggered", "initiated",
    "exceeded", "deprecated", "configured", "terminated", "dispatched",
    "exhausted", "validated", "established", "authorized", "recommended",
    "accumulated", "interrupted", "compromised", "remediated", "propagated",
    "orchestrated", "quarantined", "synchronized", "throttled", "replicated",
    "invalidated", "negotiated", "escalated", "provisioned", "decommissioned",
    "encapsulated", "materialized", "acknowledged", "insufficient", "misconfigured",
    "unresponsive", "preemption", "degradation", "fragmentation", "reconciliation",
    "serialization", "vulnerability", "compatibility", "idempotent", "transparency",
    "observability", "remediation", "notification", "classification",
]


def generate_pool(n_words, min_size=600):
    """Generate a pool of unique enum values with n_words words joined by underscores."""
    rng = random.Random(42)  # deterministic

    if n_words == 1:
        pool = []
        for i in range(min_size):
            pool.append(f"{WORDS[i % len(WORDS)]}{i // len(WORDS) if i >= len(WORDS) else ''}")
        return pool[:min_size]

    seen = set()
    pool = []
    while len(pool) < min_size:
        combo = tuple(rng.sample(WORDS, n_words))
        if combo not in seen:
            seen.add(combo)
            pool.append("_".join(combo))
    return pool


POOLS = {n: generate_pool(n) for n in [1, 2, 3, 4]}

for n in [1, 2, 3, 4]:
    print(f"{n}-word sample: {POOLS[n][0]}")

1-word sample: action
2-word sample: reconciliation_deprecated
3-word sample: reconciliation_deprecated_required
4-word sample: reconciliation_deprecated_required_remediation


In [5]:
def make_tools(pool, n_tools, enums_per_tool, values_per_enum):
    """Generate n_tools tools, each with enums_per_tool enum fields."""
    needed = n_tools * enums_per_tool * values_per_enum
    assert needed <= len(pool), f"Need {needed} values but pool has {len(pool)}"

    idx = 0
    tools = []
    for i in range(n_tools):
        props = {
            "target": {"type": "string"},
            "count": {"type": "integer"},
            "enabled": {"type": "boolean"},
        }
        for j in range(enums_per_tool):
            values = pool[idx:idx + values_per_enum]
            idx += values_per_enum
            props[f"field_{j}"] = {"type": "string", "enum": values}

        tools.append({
            "name": f"tool_{i}",
            "description": f"Tool number {i} for testing",
            "parametersJsonSchema": {
                "type": "object",
                "properties": props,
                "required": ["target", "field_0"],
            },
        })
    return tools

print(f"make_tools ready ✓")

make_tools ready ✓


## Complexity Matrix: ANY vs AUTO

For each word count (1-4), this sweeps across different schema sizes (tools × enums per tool × values per enum) and tests both ANY and AUTO modes. Longer enum strings hit the failure threshold at lower total counts.

In [6]:
def try_mode(client, fds, mode):
    """Try a generate_content call with the given mode. Returns True if successful."""
    try:
        client.models.generate_content(
            model=MODEL,
            contents="hello",
            config=types.GenerateContentConfig(
                temperature=0,
                tools=[types.Tool(function_declarations=fds)],
                tool_config=types.ToolConfig(
                    function_calling_config=types.FunctionCallingConfig(mode=mode),
                ),
            ),
        )
        return True
    except Exception:
        return False


# Configs per word count — lower thresholds for longer strings since they fail earlier
CONFIGS_BY_WORDS = {
    1: [
        (10, 4, 5),    # 200
        (15, 3, 5),    # 225
        (20, 4, 3),    # 240
        (12, 4, 5),    # 240
        (8,  6, 5),    # 240
        (15, 4, 5),    # 300
        (20, 4, 5),    # 400
    ],
    2: [
        (10, 4, 5),    # 200
        (15, 3, 5),    # 225
        (20, 4, 3),    # 240
        (12, 4, 5),    # 240
        (8,  6, 5),    # 240
        (15, 4, 5),    # 300
    ],
    3: [
        (5,  4, 5),    # 100
        (10, 4, 3),    # 120
        (8,  3, 5),    # 120
        (15, 3, 3),    # 135
        (15, 4, 3),    # 180
    ],
    4: [
        (10, 3, 3),    # 90
        (5,  4, 5),    # 100
        (10, 4, 3),    # 120
        (8,  3, 5),    # 120
        (15, 3, 3),    # 135
        (15, 4, 3),    # 180
    ],
}

for n_words in [1, 2, 3, 4]:
    pool = POOLS[n_words]
    configs = CONFIGS_BY_WORDS[n_words]
    print(f"\n{'='*60}")
    print(f"  {n_words}-word enum values (e.g. {pool[0]})")
    print(f"{'='*60}")
    print(f"{'Tools':>5} × {'Enums':>5} × {'Vals':>4} = {'Total':>5}  │  {'ANY':>6}  {'AUTO':>6}")
    print("─" * 52)

    for n_tools, enums_per_tool, values_per_enum in configs:
        total = n_tools * enums_per_tool * values_per_enum
        tools = make_tools(pool, n_tools, enums_per_tool, values_per_enum)
        fds = [types.FunctionDeclaration(**t) for t in tools]

        any_ok = try_mode(client, fds, "ANY")
        auto_ok = try_mode(client, fds, "AUTO")

        any_tag = "PASS ✓" if any_ok else "FAIL ✗"
        auto_tag = "PASS ✓" if auto_ok else "FAIL ✗"

        print(f"{n_tools:>5} × {enums_per_tool:>5} × {values_per_enum:>4} = {total:>5}  │  {any_tag:>6}  {auto_tag:>6}")


  1-word enum values (e.g. action)
Tools × Enums × Vals = Total  │     ANY    AUTO
────────────────────────────────────────────────────


   10 ×     4 ×    5 =   200  │  PASS ✓  PASS ✓


   15 ×     3 ×    5 =   225  │  PASS ✓  PASS ✓


   20 ×     4 ×    3 =   240  │  PASS ✓  PASS ✓


   12 ×     4 ×    5 =   240  │  PASS ✓  PASS ✓


...

    5 ×     4 ×    5 =   100  │  PASS ✓  PASS ✓


   10 ×     4 ×    3 =   120  │  FAIL ✗  PASS ✓


    8 ×     3 ×    5 =   120  │  PASS ✓  PASS ✓


   15 ×     3 ×    3 =   135  │  FAIL ✗  PASS ✓


   15 ×     4 ×    3 =   180  │  FAIL ✗  PASS ✓
